# 随机森林对比实验

## 目标

理解随机森林的原理和应用。

- 理解 Bagging 和随机特征选择
- 对比单棵决策树和随机森林
- 分析随机森林的参数影响
- 可视化随机森林的决策边界

## 1. 数据准备

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer, load_wine, make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_curve, auc
import pandas as pd

np.random.seed(42)

# 加载乳腺癌数据集
cancer = load_breast_cancer()
X = cancer.data
y = cancer.target
feature_names = cancer.feature_names
target_names = cancer.target_names

print(f"数据集形状: {X.shape}")
print(f"类别数量: {len(target_names)}")
print(f"类别名称: {target_names}")
print(f"类别分布: {np.bincount(y)}")
print(f"特征数量: {len(feature_names)}")

In [ ]:
# 划分训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print(f"训练集: {X_train.shape}")
print(f"测试集: {X_test.shape}")
print(f"训练集类别分布: {np.bincount(y_train)}")
print(f"测试集类别分布: {np.bincount(y_test)}")

## 2. 决策树 vs 随机森林

In [ ]:
# 训练单棵决策树
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)

# 训练随机森林
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print("决策树 vs 随机森林性能对比:")
print("="*50)
print(f"决策树:")
print(f"  训练集准确率: {dt.score(X_train, y_train):.4f}")
print(f"  测试集准确率: {accuracy_score(y_test, y_pred_dt):.4f}")

print(f"\n随机森林:")
print(f"  训练集准确率: {rf.score(X_train, y_train):.4f}")
print(f"  测试集准确率: {accuracy_score(y_test, y_pred_rf):.4f}")

## 3. 交叉验证对比

In [ ]:
# 5折交叉验证
cv_scores_dt = cross_val_score(dt, X, y, cv=5, scoring='accuracy')
cv_scores_rf = cross_val_score(rf, X, y, cv=5, scoring='accuracy')

print("5折交叉验证准确率:")
print("="*40)
print(f"决策树: {cv_scores_dt.mean():.4f} (±{cv_scores_dt.std():.4f})")
print(f"随机森林: {cv_scores_rf.mean():.4f} (±{cv_scores_rf.std():.4f})")

print(f"\n方差对比:")
print(f"决策树方差: {cv_scores_dt.var():.6f}")
print(f"随机森林方差: {cv_scores_rf.var():.6f}")
print(f"方差降低比例: {(1 - cv_scores_rf.var() / cv_scores_dt.var()) * 100:.1f}%")

## 4. 分类报告对比

In [ ]:
print("决策树分类报告:")
print(classification_report(y_test, y_pred_dt, target_names=target_names, digits=3))

print("\n随机森林分类报告:")
print(classification_report(y_test, y_pred_rf, target_names=target_names, digits=3))

## 5. 不同 n_estimators 的影响

In [ ]:
# 测试不同数量的树
n_estimators_list = [1, 5, 10, 20, 50, 100, 200, 500]

train_scores_n = []
test_scores_n = []
fit_times_n = []
predict_times_n = []

import time

for n in n_estimators_list:
    rf_temp = RandomForestClassifier(n_estimators=n, random_state=42, n_jobs=-1)
    
    start = time.time()
    rf_temp.fit(X_train, y_train)
    fit_time = time.time() - start
    
    start = time.time()
    train_score = rf_temp.score(X_train, y_train)
    predict_time = time.time() - start
    test_score = rf_temp.score(X_test, y_test)
    
    train_scores_n.append(train_score)
    test_scores_n.append(test_score)
    fit_times_n.append(fit_time)
    predict_times_n.append(predict_time)

# 可视化
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 准确率
axes[0].plot(n_estimators_list, train_scores_n, 'b-o', label='训练集', linewidth=2, markersize=6)
axes[0].plot(n_estimators_list, test_scores_n, 'r-o', label='测试集', linewidth=2, markersize=6)
axes[0].set_xlabel('树的数量 (n_estimators)')
axes[0].set_ylabel('准确率')
axes[0].set_title('准确率 vs 树的数量')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_xscale('log')

# 训练时间
axes[1].plot(n_estimators_list, fit_times_n, 'g-o', linewidth=2, markersize=6)
axes[1].set_xlabel('树的数量 (n_estimators)')
axes[1].set_ylabel('训练时间 (秒)')
axes[1].set_title('训练时间 vs 树的数量')
axes[1].grid(True, alpha=0.3)
axes[1].set_xscale('log')

# 预测时间
axes[2].plot(n_estimators_list, predict_times_n, 'm-o', linewidth=2, markersize=6)
axes[2].set_xlabel('树的数量 (n_estimators)')
axes[2].set_ylabel('预测时间 (秒)')
axes[2].set_title('预测时间 vs 树的数量')
axes[2].grid(True, alpha=0.3)
axes[2].set_xscale('log')

plt.tight_layout()
plt.show()

print(f"最佳 n_estimators: {n_estimators_list[np.argmax(test_scores_n)]}")
print(f"最佳测试集准确率: {max(test_scores_n):.4f}")

## 6. max_depth 的影响

In [ ]:
# 测试不同最大深度
max_depths = [1, 2, 3, 5, 10, 15, 20, None]

train_scores_depth = []
test_scores_depth = []

for depth in max_depths:
    rf_temp = RandomForestClassifier(n_estimators=100, max_depth=depth, random_state=42)
    rf_temp.fit(X_train, y_train)
    
    train_scores_depth.append(rf_temp.score(X_train, y_train))
    test_scores_depth.append(rf_temp.score(X_test, y_test))

# 可视化
plt.figure(figsize=(10, 6))
depth_labels = [str(d) if d is not None else 'None' for d in max_depths]
plt.plot(depth_labels, train_scores_depth, 'b-o', label='训练集', linewidth=2, markersize=6)
plt.plot(depth_labels, test_scores_depth, 'r-o', label='测试集', linewidth=2, markersize=6)
plt.xlabel('最大深度 (max_depth)')
plt.ylabel('准确率')
plt.title('准确率 vs 最大深度')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

best_idx = np.argmax(test_scores_depth)
print(f"最佳 max_depth: {max_depths[best_idx] if max_depths[best_idx] is not None else '无限制'}")
print(f"最佳测试集准确率: {test_scores_depth[best_idx]:.4f}")

## 7. max_features 的影响（随机特征选择）

In [ ]:
# 测试不同 max_features
# max_features='sqrt': 默认，分类问题使用 sqrt(n_features)
# max_features='log2': log2(n_features)
# max_features=None: 使用全部特征
max_features_options = ['sqrt', 'log2', None, 0.3, 0.5, 0.7]

train_scores_feat = []
test_scores_feat = []

for feat in max_features_options:
    rf_temp = RandomForestClassifier(n_estimators=100, max_features=feat, random_state=42)
    rf_temp.fit(X_train, y_train)
    
    train_scores_feat.append(rf_temp.score(X_train, y_train))
    test_scores_feat.append(rf_temp.score(X_test, y_test))

# 可视化
plt.figure(figsize=(10, 6))
feat_labels = [str(f) if f is not None else 'None' for f in max_features_options]
plt.bar(range(len(max_features_options)), train_scores_feat, alpha=0.6, label='训练集', color='blue')
plt.bar(range(len(max_features_options)), test_scores_feat, alpha=0.6, label='测试集', color='red')
plt.xlabel('max_features')
plt.ylabel('准确率')
plt.title('准确率 vs max_features')
plt.xticks(range(len(max_features_options)), feat_labels)
plt.legend()
plt.grid(True, alpha=0.3, axis='y')
plt.show()

best_idx = np.argmax(test_scores_feat)
print(f"最佳 max_features: {max_features_options[best_idx]}")
print(f"最佳测试集准确率: {test_scores_feat[best_idx]:.4f}")

## 8. 特征重要性对比

In [ ]:
# 获取特征重要性
dt_importance = dt.feature_importances_
rf_importance = rf.feature_importances_

# 创建DataFrame对比
importance_df = pd.DataFrame({
    '特征': feature_names,
    '决策树重要性': dt_importance,
    '随机森林重要性': rf_importance
})

# 按随机森林重要性排序
importance_df = importance_df.sort_values('随机森林重要性', ascending=False)

# 显示前15个最重要的特征
print("Top 15 最重要的特征（按随机森林重要性排序）:")
print(importance_df.head(15).to_string(index=False))

In [ ]:
# 可视化特征重要性对比
top_n = 10
top_features = importance_df.head(top_n)

plt.figure(figsize=(14, 6))
x = np.arange(top_n)
width = 0.35

plt.bar(x - width/2, top_features['决策树重要性'], width, label='决策树', alpha=0.7)
plt.bar(x + width/2, top_features['随机森林重要性'], width, label='随机森林', alpha=0.7)

plt.xlabel('特征')
plt.ylabel('重要性')
plt.title(f'Top {top_n} 特征重要性对比')
plt.xticks(x, top_features['特征'], rotation=45, ha='right')
plt.legend()
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 9. 混淆矩阵对比

In [ ]:
# 计算混淆矩阵
cm_dt = confusion_matrix(y_test, y_pred_dt)
cm_rf = confusion_matrix(y_test, y_pred_rf)

# 可视化
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 决策树
im1 = axes[0].imshow(cm_dt, interpolation='nearest', cmap=plt.cm.Blues)
axes[0].set_title('决策树混淆矩阵')
tick_marks = np.arange(len(target_names))
axes[0].set_xticks(tick_marks)
axes[0].set_yticks(tick_marks)
axes[0].set_xticklabels(target_names)
axes[0].set_yticklabels(target_names)
axes[0].set_xlabel('预测标签')
axes[0].set_ylabel('真实标签')

for i in range(cm_dt.shape[0]):
    for j in range(cm_dt.shape[1]):
        axes[0].text(j, i, format(cm_dt[i, j], 'd'),
                 horizontalalignment="center",
                 color="white" if cm_dt[i, j] > cm_dt.max() / 2 else "black")

plt.colorbar(im1, ax=axes[0])

# 随机森林
im2 = axes[1].imshow(cm_rf, interpolation='nearest', cmap=plt.cm.Blues)
axes[1].set_title('随机森林混淆矩阵')
axes[1].set_xticks(tick_marks)
axes[1].set_yticks(tick_marks)
axes[1].set_xticklabels(target_names)
axes[1].set_yticklabels(target_names)
axes[1].set_xlabel('预测标签')
axes[1].set_ylabel('真实标签')

for i in range(cm_rf.shape[0]):
    for j in range(cm_rf.shape[1]):
        axes[1].text(j, i, format(cm_rf[i, j], 'd'),
                 horizontalalignment="center",
                 color="white" if cm_rf[i, j] > cm_rf.max() / 2 else "black")

plt.colorbar(im2, ax=axes[1])
plt.tight_layout()
plt.show()

## 10. ROC 曲线对比

In [ ]:
# 获取预测概率
y_prob_dt = dt.predict_proba(X_test)[:, 1]
y_prob_rf = rf.predict_proba(X_test)[:, 1]

# 计算ROC曲线
fpr_dt, tpr_dt, _ = roc_curve(y_test, y_prob_dt)
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_prob_rf)

roc_auc_dt = auc(fpr_dt, tpr_dt)
roc_auc_rf = auc(fpr_rf, tpr_rf)

# 可视化
plt.figure(figsize=(10, 8))
plt.plot(fpr_dt, tpr_dt, color='blue', lw=2, label=f'决策树 (AUC = {roc_auc_dt:.3f})')
plt.plot(fpr_rf, tpr_rf, color='red', lw=2, label=f'随机森林 (AUC = {roc_auc_rf:.3f})')
plt.plot([0, 1], [0, 1], color='gray', lw=2, linestyle='--', label='随机猜测')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('假阳性率 (FPR)')
plt.ylabel('真阳性率 (TPR)')
plt.title('ROC 曲线对比')
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)
plt.show()

print(f"决策树 AUC: {roc_auc_dt:.4f}")
print(f"随机森林 AUC: {roc_auc_rf:.4f}")

## 11. 决策边界可视化（2D数据）

In [ ]:
# 生成2D数据
X_2d, y_2d = make_classification(
    n_samples=300,
    n_features=2,
    n_redundant=0,
    n_informative=2,
    random_state=42,
    n_clusters_per_class=2
)

# 训练模型
dt_2d = DecisionTreeClassifier(random_state=42)
rf_2d = RandomForestClassifier(n_estimators=100, random_state=42)

dt_2d.fit(X_2d, y_2d)
rf_2d.fit(X_2d, y_2d)

# 绘制决策边界
def plot_decision_boundary_2d(model, X, y, title):
    h = 0.02
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    Z = model.predict_proba(np.c_[xx.ravel(), yy.ravel()])[:, 1]
    Z = Z.reshape(xx.shape)
    
    plt.figure(figsize=(10, 8))
    plt.contourf(xx, yy, Z, alpha=0.3, levels=np.linspace(0, 1, 11), cmap=plt.cm.RdYlBu)
    plt.contour(xx, yy, Z, levels=[0.5], linewidths=2, colors='black')
    plt.scatter(X[y == 0][:, 0], X[y == 0][:, 1], c='blue', label='类别 0', alpha=0.6)
    plt.scatter(X[y == 1][:, 0], X[y == 1][:, 1], c='red', label='类别 1', alpha=0.6)
    plt.xlabel('特征 1')
    plt.ylabel('特征 2')
    plt.title(f'{title} (准确率: {model.score(X, y):.3f})')
    plt.legend()
    plt.colorbar(label='类别 1 的概率')
    plt.grid(True, alpha=0.3)
    plt.show()

plot_decision_boundary_2d(dt_2d, X_2d, y_2d, '决策树')
plot_decision_boundary_2d(rf_2d, X_2d, y_2d, '随机森林')

## 12. OOB 误差（Out-of-Bag Error）

In [ ]:
# 使用 OOB 评估随机森林
rf_oob = RandomForestClassifier(
    n_estimators=100,
    oob_score=True,
    random_state=42
)
rf_oob.fit(X_train, y_train)

print(f"OOB 得分: {rf_oob.oob_score_:.4f}")
print(f"训练集得分: {rf_oob.score(X_train, y_train):.4f}")
print(f"测试集得分: {rf_oob.score(X_test, y_test):.4f}")

# 使用 OOB 误差曲线
oob_errors = []
n_estimators_range = range(10, 200, 10)

for n in n_estimators_range:
    rf_temp = RandomForestClassifier(
        n_estimators=n,
        oob_score=True,
        random_state=42
    )
    rf_temp.fit(X_train, y_train)
    oob_errors.append(1 - rf_temp.oob_score_)

plt.figure(figsize=(10, 6))
plt.plot(n_estimators_range, oob_errors, 'b-o', linewidth=2, markersize=6)
plt.xlabel('树的数量')
plt.ylabel('OOB 误差')
plt.title('OOB 误差 vs 树的数量')
plt.grid(True, alpha=0.3)
plt.show()

best_n = n_estimators_range[np.argmin(oob_errors)]
print(f"基于 OOB 的最佳 n_estimators: {best_n}")
print(f"最小 OOB 误差: {min(oob_errors):.4f}")

## 13. 网格搜索调参

In [ ]:
# 定义参数网格
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 10, None],
    'max_features': ['sqrt', 'log2'],
    'min_samples_split': [2, 5, 10]
}

# 网格搜索
grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=0
)

grid_search.fit(X_train, y_train)

print("最佳参数:")
for param, value in grid_search.best_params_.items():
    print(f"  {param}: {value}")

print(f"\n最佳交叉验证得分: {grid_search.best_score_:.4f}")
print(f"测试集准确率: {grid_search.score(X_test, y_test):.4f}")
print(f"\n与默认参数随机森林相比:")
print(f"  默认参数测试集准确率: {rf.score(X_test, y_test):.4f}")
print(f"  改善: {grid_search.score(X_test, y_test) - rf.score(X_test, y_test):.4f}")

## 14. 单棵树的可视化（从随机森林中）

In [ ]:
from sklearn.tree import plot_tree

# 从随机森林中抽取一棵树
single_tree = rf.estimators_[0]

# 可视化
plt.figure(figsize=(20, 10))
plot_tree(single_tree,
          feature_names=feature_names,
          class_names=target_names,
          max_depth=3,
          filled=True,
          rounded=True,
          fontsize=8)
plt.title('随机森林中的单棵决策树（前3层）', fontsize=16)
plt.tight_layout()
plt.show()

print(f"这棵树的深度: {single_tree.get_depth()}")
print(f"这棵树的叶节点数: {single_tree.get_n_leaves()}")

## 15. 随机森林在不同数据集上的表现

In [ ]:
# 在不同数据集上测试
datasets = [
    ('Iris', load_iris(), 'multi'),
    ('Breast Cancer', load_breast_cancer(), 'binary'),
    ('Wine', load_wine(), 'multi')
]

results = []

for name, dataset, task_type in datasets:
    X_data, y_data = dataset.data, dataset.target
    
    X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(
        X_data, y_data, test_size=0.3, random_state=42, stratify=y_data
    )
    
    # 训练决策树
    dt_temp = DecisionTreeClassifier(random_state=42)
    dt_temp.fit(X_train_d, y_train_d)
    dt_score = dt_temp.score(X_test_d, y_test_d)
    
    # 训练随机森林
    rf_temp = RandomForestClassifier(n_estimators=100, random_state=42)
    rf_temp.fit(X_train_d, y_train_d)
    rf_score = rf_temp.score(X_test_d, y_test_d)
    
    results.append({
        '数据集': name,
        '类型': task_type,
        '样本数': len(X_data),
        '特征数': X_data.shape[1],
        '类别数': len(np.unique(y_data)),
        '决策树': dt_score,
        '随机森林': rf_score,
        '改善': rf_score - dt_score
    })

# 显示结果
results_df = pd.DataFrame(results)
print("决策树 vs 随机森林在不同数据集上的表现:")
print(results_df.round(4).to_string(index=False))

## 总结

### 随机森林 vs 决策树

| 特性 | 决策树 | 随机森林 |
|------|--------|----------|
| 方差 | 高 | 低 |
| 偏差 | 低 | 略高 |
| 过拟合 | 容易 | 不容易 |
| 可解释性 | 高 | 低 |
| 训练速度 | 快 | 慢 |
| 预测速度 | 快 | 中等 |

### 随机森林的关键特性

1. **Bagging（Bootstrap Aggregating）**:
   - 每棵树使用不同的 bootstrap 样本
   - 降低模型的方差

2. **随机特征选择**:
   - 每个节点只考虑特征的子集
   - 增加树之间的多样性
   - 默认 `max_features='sqrt'`

3. **OOB（Out-of-Bag）评估**:
   - 使用未被选中的样本进行验证
   - 类似交叉验证的效果
   - 不需要额外的验证集

### 关键参数

- `n_estimators`: 树的数量（越多越好，但计算成本高）
- `max_depth`: 树的最大深度（控制过拟合）
- `max_features`: 每个节点考虑的特征数
- `min_samples_split`: 分裂所需的最小样本数
- `oob_score`: 是否使用 OOB 评估

### 实践建议

- 默认从 100 棵树开始（`n_estimators=100`）
- 使用 OOB 得分估计性能
- 使用网格搜索调参
- 对于高维数据，考虑减少 `max_features`
- 优先使用随机森林而非单棵决策树
- 可以通过特征重要性进行特征选择